# Week 2 - Hypothesis Testing

Uses `src/statistics.py`'s `StatisticalAnalyzer` on the Week 1 cleaned heart disease data (`data/processed/cleaned_data.csv`).

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.data_loader import DataLoader
from src.statistics import StatisticalAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data
loader = DataLoader()
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv")

# Re-apply categorical dtype for the clinical flag/category columns
# (dtype doesn't survive a plain CSV round-trip)
for col in df.columns:
    if df[col].nunique() < 10:
        df[col] = df[col].astype('category')

# Initialize analyzer
analyzer = StatisticalAnalyzer(df)

print("=" * 60)
print("HYPOTHESIS TESTING")
print("=" * 60)

In [2]:
# ==================== 1. T-TESTS ====================
print("\n T-TESTS")

# One-sample t-test: is average cholesterol significantly different
# from 200 mg/dl (the clinical "borderline high" threshold)?
result = analyzer.t_test('chol', 200)
print(f"\nOne-sample t-test on chol (vs. clinical threshold of 200 mg/dl):")
print(f"  Mean: {result['mean']:.3f}")
print(f"  t-statistic: {result['statistic']:.3f}")
print(f"  p-value: {result['p_value']:.4e}")
print(f"  Significant: {result['significant']}")
print(f"  Interpretation: {result['interpretation']}")

# Independent t-test: compare maximum heart rate (thalach) between
# patients with and without a heart disease diagnosis
result = analyzer.t_test('thalach', 'target', group1=0, group2=1)
print(f"\nIndependent t-test: thalach by target (0 = no disease, 1 = disease)")
print(f"  Mean (target=0): {result['mean1']:.3f}")
print(f"  Mean (target=1): {result['mean2']:.3f}")
print(f"  t-statistic: {result['statistic']:.3f}")
print(f"  p-value: {result['p_value']:.4e}")
print(f"  Significant: {result['significant']}")
print(f"  Interpretation: {result['interpretation']}")

# Independent t-test: compare ST depression (oldpeak) between
# patients with and without a heart disease diagnosis
result2 = analyzer.t_test('oldpeak', 'target', group1=0, group2=1)
print(f"\nIndependent t-test: oldpeak by target (0 = no disease, 1 = disease)")
print(f"  Mean (target=0): {result2['mean1']:.3f}")
print(f"  Mean (target=1): {result2['mean2']:.3f}")
print(f"  t-statistic: {result2['statistic']:.3f}")
print(f"  p-value: {result2['p_value']:.4e}")
print(f"  Significant: {result2['significant']}")
print(f"  Interpretation: {result2['interpretation']}")


 T-TESTS

One-sample t-test on chol (vs. clinical threshold of 200 mg/dl):
  Mean: 245.377
  t-statistic: 16.606
  p-value: 2.0659e-44
  Significant: True
  Interpretation: Mean significantly differs from 200

Independent t-test: thalach by target (0 = no disease, 1 = disease)
  Mean (target=0): 139.197
  Mean (target=1): 158.378
  t-statistic: -7.922
  p-value: 6.0210e-14
  Significant: True
  Interpretation: Significant difference in thalach between target=0 and target=1

Independent t-test: oldpeak by target (0 = no disease, 1 = disease)
  Mean (target=0): 1.554
  Mean (target=1): 0.585
  t-statistic: 8.071
  p-value: 4.1925e-14
  Significant: True
  Interpretation: Significant difference in oldpeak between target=0 and target=1

In [3]:
# ==================== 2. ANOVA ====================
print("\n ONE-WAY ANOVA")

# Does maximum heart rate differ across chest pain types?
result3 = analyzer.anova_test('thalach', 'cp')
print(f"\nANOVA: Maximum Heart Rate (thalach) by Chest Pain Type (cp)")
print(f"  Groups: {result3['groups']}")
print(f"  F-statistic: {result3['f_statistic']:.3f}")
print(f"  p-value: {result3['p_value']:.4e}")
print(f"  Significant: {result3['significant']}")
if result3['significant']:
    print("  Interpretation: At least one chest pain type has a significantly different average maximum heart rate")

# Does cholesterol differ across chest pain types?
result4 = analyzer.anova_test('chol', 'cp')
print(f"\nANOVA: Cholesterol (chol) by Chest Pain Type (cp)")
print(f"  Groups: {result4['groups']}")
print(f"  F-statistic: {result4['f_statistic']:.3f}")
print(f"  p-value: {result4['p_value']:.4e}")
print(f"  Significant: {result4['significant']}")


 ONE-WAY ANOVA

ANOVA: Maximum Heart Rate (thalach) by Chest Pain Type (cp)
  Groups: 4
  F-statistic: 17.663
  p-value: 1.4029e-10
  Significant: True
  Interpretation: At least one chest pain type has a significantly different average maximum heart rate

ANOVA: Cholesterol (chol) by Chest Pain Type (cp)
  Groups: 4
  F-statistic: 0.806
  p-value: 4.9118e-01
  Significant: False

In [4]:
# ==================== 3. CHI-SQUARE TEST ====================
print("\n CHI-SQUARE TEST")

# For datasets with categorical variables - automatically pick the
# first two (these are 'target' and 'sex' after Week 1's column reorder)
categorical_cols = list(df.select_dtypes(include=['object', 'category']).columns)
col1, col2 = categorical_cols[0], categorical_cols[1]
result5 = analyzer.chi_square_test(col1, col2)
print(f"\nChi-square Test: {col1} vs {col2}")
print(f"  chi2-statistic: {result5['chi2_statistic']:.3f}")
print(f"  p-value: {result5['p_value']:.4e}")
print(f"  degrees of freedom: {result5['degrees_of_freedom']}")
print(f"  Significant: {result5['significant']}")
print(f"  Interpretation: {result5['interpretation']}")

# A second, clinically relevant chi-square test: is heart disease
# diagnosis independent of exercise-induced angina?
result6 = analyzer.chi_square_test('target', 'exang')
print(f"\nChi-square Test: target vs exang")
print(f"  chi2-statistic: {result6['chi2_statistic']:.3f}")
print(f"  p-value: {result6['p_value']:.4e}")
print(f"  degrees of freedom: {result6['degrees_of_freedom']}")
print(f"  Significant: {result6['significant']}")
print(f"  Interpretation: {result6['interpretation']}")


 CHI-SQUARE TEST

Chi-square Test: target vs sex
  chi2-statistic: 23.084
  p-value: 1.5509e-06
  degrees of freedom: 1
  Significant: True
  Interpretation: Variables are dependent

Chi-square Test: target vs exang
  chi2-statistic: 55.456
  p-value: 9.5565e-14
  degrees of freedom: 1
  Significant: True
  Interpretation: Variables are dependent

In [5]:
# ==================== 4. SUMMARY OF ALL TESTS ====================
print("\n" + "=" * 60)
print("SUMMARY OF HYPOTHESIS TESTS")
print("=" * 60)

summary_rows = []
r = analyzer.t_test('chol', 200)
summary_rows.append({'Test': 'One-sample t-test', 'Variable 1': 'chol', 'Variable 2': '200 (threshold)',
                      'Statistic': r['statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.t_test('thalach', 'target', group1=0, group2=1)
summary_rows.append({'Test': 'Independent t-test', 'Variable 1': 'thalach', 'Variable 2': 'target',
                      'Statistic': r['statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.t_test('oldpeak', 'target', group1=0, group2=1)
summary_rows.append({'Test': 'Independent t-test', 'Variable 1': 'oldpeak', 'Variable 2': 'target',
                      'Statistic': r['statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.anova_test('thalach', 'cp')
summary_rows.append({'Test': 'One-way ANOVA', 'Variable 1': 'thalach', 'Variable 2': 'cp',
                      'Statistic': r['f_statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.anova_test('chol', 'cp')
summary_rows.append({'Test': 'One-way ANOVA', 'Variable 1': 'chol', 'Variable 2': 'cp',
                      'Statistic': r['f_statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.chi_square_test('target', 'sex')
summary_rows.append({'Test': 'Chi-square', 'Variable 1': 'target', 'Variable 2': 'sex',
                      'Statistic': r['chi2_statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})
r = analyzer.chi_square_test('target', 'exang')
summary_rows.append({'Test': 'Chi-square', 'Variable 1': 'target', 'Variable 2': 'exang',
                      'Statistic': r['chi2_statistic'], 'P-value': r['p_value'], 'Significant': r['significant']})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../reports/hypothesis_tests_summary.csv', index=False)
print(summary_df.to_string())
print("\nHypothesis test results saved to 'reports/hypothesis_tests_summary.csv'")


SUMMARY OF HYPOTHESIS TESTS
                 Test Variable 1       Variable 2  Statistic       P-value  Significant
0   One-sample t-test       chol  200 (threshold)  16.606116  2.065924e-44         True
1  Independent t-test    thalach           target  -7.921881  6.021019e-14         True
2  Independent t-test    oldpeak           target   8.070717  4.192516e-14         True
3       One-way ANOVA    thalach               cp  17.662978  1.402854e-10         True
4       One-way ANOVA       chol               cp   0.806320  4.911769e-01        False
5          Chi-square     target              sex  23.083879  1.550855e-06         True
6          Chi-square     target            exang  55.456203  9.556466e-14         True

Hypothesis test results saved to 'reports/hypothesis_tests_summary.csv'